In [1]:
import pandas as pd 
import numpy as np
import joblib
import time
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    f1_score,
    recall_score,
    confusion_matrix,
    roc_auc_score
)
from xgboost import XGBClassifier
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.ensemble import RandomForestClassifier

In [2]:
## Load augmnented datasets
train = pd.read_csv('../datasets/train_tvae.csv')
test = pd.read_csv('../datasets/test_shap_66.csv')

X_train = train.drop(['Label'], axis=1)
y_train = train['Label']
X_test = test.drop(['Label'], axis=1)
y_test = test['Label']
y_test = pd.Series(y_test)
y_train = pd.Series(y_train)

## LSTM

In [3]:
# ============================
# LSTM cho Tabular — IEC/ICS preset — KHÔNG focus lớp nào
#  - Stratified split (VAL 15%)
#  - StandardScaler
#  - DataLoader (shuffle, không sampler/boost)
#  - LSTM (2 tầng, BiLSTM) + AttnPooling
#  - Mixup công bằng cho mọi lớp
#  - EMA (float-only) device-safe
#  - Early-stopping theo Macro-F1 (VAL)
#  - Đánh giá: Acc, Macro-F1, per-class metrics
# ============================
import math, numpy as np, pandas as pd, torch, torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report, precision_recall_fscore_support

# ----- Device & seed -----
device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(42); np.random.seed(42)

# ===== Dữ liệu =====
assert "Label" in train.columns and "Label" in test.columns, "Thiếu cột Label trong train/test"
X_full = train.drop(columns=["Label"]).values
y_full = train["Label"].astype(int).values
X_test = test.drop(columns=["Label"]).values
y_test = test["Label"].astype(int).values

num_classes  = int(max(y_full.max(), y_test.max())) + 1
num_features = X_full.shape[1]

# ===== 1) Stratified split: train_in / val (15%) =====
sss = StratifiedShuffleSplit(n_splits=1, test_size=0.15, random_state=42)
train_idx, val_idx = next(sss.split(X_full, y_full))
X_tr_in, y_tr_in = X_full[train_idx], y_full[train_idx]
X_val,   y_val   = X_full[val_idx],   y_full[val_idx]

# ===== 2) Scaler =====
scaler = StandardScaler()
X_tr_in_sc = scaler.fit_transform(X_tr_in)
X_val_sc   = scaler.transform(X_val)
X_test_sc  = scaler.transform(X_test)

# ===== 3) DataLoaders =====
Xtr_t = torch.tensor(X_tr_in_sc, dtype=torch.float32)
ytr_t = torch.tensor(y_tr_in,    dtype=torch.long)
Xva_t = torch.tensor(X_val_sc,   dtype=torch.float32)
yva_t = torch.tensor(y_val,      dtype=torch.long)
Xte_t = torch.tensor(X_test_sc,  dtype=torch.float32)
yte_t = torch.tensor(y_test,     dtype=torch.long)

batch_size = 2048
train_loader = DataLoader(TensorDataset(Xtr_t, ytr_t), batch_size=batch_size, shuffle=True,  pin_memory=True)
val_loader   = DataLoader(TensorDataset(Xva_t, yva_t), batch_size=4096, shuffle=False, pin_memory=True)
test_loader  = DataLoader(TensorDataset(Xte_t, yte_t), batch_size=4096, shuffle=False, pin_memory=True)

# ===== 4) Tiện ích: chia feature thành "chuỗi" =====
# Ta coi features như một chuỗi gồm S steps, mỗi step có input_size = STEP_DIM
# (pad 0 nếu F không chia hết)
STEP_DIM = 16  # bạn có thể thử 8/16/32
def as_sequence(x_np: np.ndarray, step_dim: int = STEP_DIM):
    F = x_np.shape[1]
    S = int(np.ceil(F / step_dim))
    pad = S * step_dim - F
    if pad > 0:
        x_np = np.pad(x_np, ((0,0),(0,pad)), mode="constant")
    return x_np.reshape(-1, S, step_dim), S

# ===== 5) Mô hình LSTM + AttnPooling =====
class AttnPool(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.proj = nn.Linear(d, 1)
    def forward(self, H):        # H: [B,S,D]
        score = self.proj(H).squeeze(-1)           # [B,S]
        w = torch.softmax(score, dim=1)            # [B,S]
        pooled = (H * w.unsqueeze(-1)).sum(1)      # [B,D]
        return pooled

class LSTMTabular(nn.Module):
    def __init__(self, step_dim, hidden=256, layers=2, n_classes=10, dropout=0.2, bidir=True):
        super().__init__()
        self.step_dim = step_dim
        self.in_norm  = nn.LayerNorm(step_dim)
        self.lstm = nn.LSTM(
            input_size=step_dim,
            hidden_size=hidden,
            num_layers=layers,
            batch_first=True,
            dropout=dropout if layers > 1 else 0.0,
            bidirectional=bidir
        )
        d_out = hidden * (2 if bidir else 1)
        self.pool = AttnPool(d_out)
        self.head = nn.Sequential(
            nn.Linear(d_out, d_out//2), nn.ReLU(), nn.Dropout(0.20),
            nn.Linear(d_out//2, n_classes)
        )
    def forward(self, x):                       # x: [B, F]
        B, F = x.shape
        S = int(math.ceil(F / self.step_dim))
        pad = S * self.step_dim - F
        if pad > 0:
            x = torch.nn.functional.pad(x, (0, pad), value=0.0)
        x = x.view(B, S, self.step_dim)         # [B,S,step_dim]
        x = self.in_norm(x)
        H, _ = self.lstm(x)                     # [B,S,D]
        z = self.pool(H)                        # [B,D]
        return self.head(z)                     # [B,C]

model = LSTMTabular(step_dim=STEP_DIM, hidden=256, layers=2, n_classes=num_classes, dropout=0.15, bidir=True).to(device)

# ===== 6) Loss/optim/scheduler + mixup (công bằng) =====
criterion = nn.CrossEntropyLoss(label_smoothing=0.05)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=2e-4)

total_epochs = 120
warmup_epochs = 19
steps_per_epoch = max(1, len(train_loader))
def lr_lambda(step):
    warmup_steps = warmup_epochs * steps_per_epoch
    total_steps  = total_epochs * steps_per_epoch
    if step < warmup_steps: return (step + 1) / warmup_steps
    progress = (step - warmup_steps) / max(1, (total_steps - warmup_steps))
    return 0.5 * (1 + math.cos(math.pi * progress))
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

def mixup_pair(x, y, alpha=0.10):
    if alpha <= 0: return x, y, 1.0, None
    lam  = np.random.beta(alpha, alpha)
    perm = torch.randperm(x.size(0), device=x.device)
    x_m  = lam * x + (1 - lam) * x[perm]
    y_perm = y[perm]
    return x_m, y, lam, y_perm

# ===== 7) EMA (float-only) — device-safe =====
ema_decay = 0.995
ema_state = {}
for k, v in model.state_dict().items():
    if v.is_floating_point():
        ema_state[k] = v.detach().clone().to(device=v.device, dtype=v.dtype)

def ema_update(model, ema_state, decay: float):
    with torch.no_grad():
        for k, v in model.state_dict().items():
            if not v.is_floating_point(): continue
            if (ema_state[k].device != v.device) or (ema_state[k].dtype != v.dtype):
                ema_state[k] = ema_state[k].to(device=v.device, dtype=v.dtype)
            ema_state[k].mul_((decay)).add_(v.detach(), alpha=1.0 - decay)

def load_ema_state(model, ema_state):
    current = model.state_dict(); merged = {}
    for k, v in current.items():
        merged[k] = ema_state.get(k, v).to(device=v.device, dtype=v.dtype)
    model.load_state_dict(merged, strict=True)

# ===== 8) Train + Early-stopping theo Macro-F1 (VAL) =====
best_macro_f1, best_ema_snapshot, wait, patience = -1.0, None, 0, 20
global_step = 0

for epoch in range(total_epochs):
    model.train()
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        xb_m, ya, lam, yb_ = mixup_pair(xb, yb, alpha=0.10)
        logits = model(xb_m)
        loss = lam * criterion(logits, ya) + (1 - lam) * criterion(logits, yb_) if yb_ is not None else criterion(logits, ya)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 3.0)
        optimizer.step(); scheduler.step()
        ema_update(model, ema_state, ema_decay)
        global_step += 1

    # Eval VAL với EMA
    saved_live = {k: v.detach().clone() for k, v in model.state_dict().items()}
    load_ema_state(model, ema_state)
    model.eval()
    with torch.no_grad():
        logits_val = []
        for xb, _ in val_loader:
            logits_val.append(model(xb.to(device)).cpu())
        logits_val = torch.cat(logits_val, 0).numpy()
    pred_val = logits_val.argmax(1)
    macro_f1 = f1_score(y_val, pred_val, average='macro', zero_division=0)

    if macro_f1 > best_macro_f1:
        best_macro_f1, wait = macro_f1, 0
        best_ema_snapshot = {k: v.detach().cpu().clone() for k, v in ema_state.items()}
    else:
        wait += 1

    model.load_state_dict(saved_live, strict=True)
    if wait >= patience:
        print(f"[Early-stop] epoch {epoch+1} | best VAL Macro-F1 = {best_macro_f1:.4f}")
        break

# Nạp EMA tốt nhất
if best_ema_snapshot is not None:
    ema_state = {k: t.clone().to(device) for k, t in best_ema_snapshot.items()}
    load_ema_state(model, ema_state)
model.eval()

# ===== 9) SUY LUẬN TEST =====
with torch.no_grad():
    logits_test = []
    for xb, _ in test_loader:
        logits_test.append(model(xb.to(device)).cpu())
    logits_test = torch.cat(logits_test, 0).numpy()

y_pred = logits_test.argmax(1)

# ===== 10) Đánh giá =====
acc = accuracy_score(y_test, y_pred)
macro_f1_all = f1_score(y_test, y_pred, average='macro', zero_division=0)

cm = confusion_matrix(y_test, y_pred, labels=np.arange(num_classes))
print(f"\nLSTM-Tabular — Acc: {acc:.4f} | Macro-F1(all): {macro_f1_all:.4f}")
print("\nClassification report:")
print(classification_report(y_test, y_pred, digits=4))

labels_arr = np.arange(num_classes)
prec, rec, f1c, support = precision_recall_fscore_support(
    y_test, y_pred, labels=labels_arr, zero_division=0
)

# FPR per class
fpr_per_class = []
for i in labels_arr:
    TP = cm[i, i]
    FN = cm[i, :].sum() - TP
    FP = cm[:, i].sum() - TP
    TN = cm.sum() - (TP + FN + FP)
    fpr_i = FP / (FP + TN) if (FP + TN) > 0 else 0.0
    fpr_per_class.append(fpr_i)

macro_precision = float(np.mean(prec))
macro_recall    = float(np.mean(rec))
macro_fpr       = float(np.mean(fpr_per_class))

print("\n==== Macro metrics (all classes) ====")
print(f"Macro-Precision: {macro_precision:.4f}")
print(f"Macro-Recall/TPR: {macro_recall:.4f}")
print(f"Macro-FPR: {macro_fpr:.4f}")

per_class_df = pd.DataFrame({
    "class": labels_arr,
    "precision": prec,
    "recall_TPR": rec,
    "f1": f1c,
    "FPR": fpr_per_class,
    "support": support,
}).round(4)
print("\n==== Per-class metrics (all classes) ====")
print(per_class_df)


LSTM-Tabular — Acc: 0.6810 | Macro-F1(all): 0.6675

Classification report:
              precision    recall  f1-score   support

           0     0.9883    1.0000    0.9941       169
           1     0.8187    0.8284    0.8235       169
           2     0.7945    0.6864    0.7365       169
           3     0.2787    0.1006    0.1478       169
           4     0.4054    0.3550    0.3785       169
           5     0.6575    0.8521    0.7423       169
           6     0.4480    0.6627    0.5346       169
           7     0.5983    0.8284    0.6948       169
           8     0.8095    0.7041    0.7532       169
           9     0.5988    0.5740    0.5861       169
          10     0.6601    0.5976    0.6273       169
          11     1.0000    0.9822    0.9910       169

    accuracy                         0.6810      2028
   macro avg     0.6715    0.6810    0.6675      2028
weighted avg     0.6715    0.6810    0.6675      2028


==== Macro metrics (all classes) ====
Macro-Precision: 0

In [4]:
# Save LSTM checkpoint (compatible with LSTMWrapper.from_checkpoint)
import torch


checkpoint = {
    'state_dict': model.state_dict(),
    'scaler': scaler,
    'input_dim': num_features,
    'num_classes': num_classes,
    'step_dim': STEP_DIM,
    'hidden': 256,
    'layers': 2,
    'dropout': 0.15,
    'bidir': True,
}

save_path = './models/framework_lstm_TVAE.pth'
torch.save(checkpoint, save_path)
print(f"LSTM checkpoint saved to {save_path}")
print(f"  input_dim={num_features}, num_classes={num_classes}, step_dim={STEP_DIM}")

LSTM checkpoint saved to ./models/framework_lstm_TVAE.pth
  input_dim=66, num_classes=12, step_dim=16


## ResDNN

In [5]:
# ============================
# DNN Tabular (ResNet-style) — ICS preset (EMA-float only) — SAFE DEVICE
# Tập trung lớp 7 & 10 (không đổi hyper-params)
# Tích hợp:
#  - CB-Focal + class weights (Effective Number)
#  - Sampler boost cho lớp 7/10
#  - Mixup giảm khi batch có 7/10
#  - EMA an toàn thiết bị (device-safe)
#  - Early-stopping theo Macro-F1 (VAL)
#  - Temperature scaling (VAL)
#  - Logit-bias search (VAL) để đẩy 7/10
#  - (Tuỳ chọn) Finetune head tách rời
#  - (Tuỳ chọn) OvR blend cho 7/10
# ============================
import math, numpy as np, pandas as pd, torch, torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader, WeightedRandomSampler
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report, precision_recall_fscore_support

# ===== CỜ TUỲ CHỌN =====
ENABLE_HEAD_FINETUNE = True      # finetune riêng head vài epoch để nhạy với 7/10
ENABLE_OVR_BLEND     = False     # OvR cho 7/10 (LogReg) rồi blend vào xác suất DNN
OPT_BIAS_FOR_TARGETS = False     # True: tối ưu macro-F1 chỉ trên {7,10} khi chọn T & bias

# ===== Device & seed =====
device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(42); np.random.seed(42)

# ===== Dữ liệu =====
assert "Label" in train.columns and "Label" in test.columns, "Thiếu cột Label trong train/test!"
X_full = train.drop(columns=["Label"]).values
y_full = train["Label"].astype(int).values
X_test = test.drop(columns=["Label"]).values
y_test = test["Label"].astype(int).values

num_classes  = int(max(y_full.max(), y_test.max())) + 1
num_features = X_full.shape[1]

# ===== 1) Stratified split: train_in / val (15%) =====
sss = StratifiedShuffleSplit(n_splits=1, test_size=0.15, random_state=42)
train_idx, val_idx = next(sss.split(X_full, y_full))
X_tr_in, y_tr_in = X_full[train_idx], y_full[train_idx]
X_val,   y_val   = X_full[val_idx],   y_full[val_idx]

# ===== 2) Scaler =====
scaler = StandardScaler()
X_tr_in_sc = scaler.fit_transform(X_tr_in)
X_val_sc   = scaler.transform(X_val)
X_test_sc  = scaler.transform(X_test)

# ===== 3) Sampler cân bằng + boost 7/10 =====
counts = np.bincount(y_tr_in, minlength=num_classes).astype(np.float32)
inv = counts.sum() / (counts + 1e-9)
inv /= inv.mean()
boost = np.ones(num_classes, dtype=np.float32)
boost[[7,10]] = 1.6  # tăng tần suất gặp 7/10 (giữ hệ số)
sample_w = (inv * boost)[y_tr_in]
sampler  = WeightedRandomSampler(weights=sample_w, num_samples=len(sample_w), replacement=True)

# ===== 4) DataLoaders =====
Xtr_t = torch.tensor(X_tr_in_sc, dtype=torch.float32)
ytr_t = torch.tensor(y_tr_in,    dtype=torch.long)
Xva_t = torch.tensor(X_val_sc,   dtype=torch.float32)
yva_t = torch.tensor(y_val,      dtype=torch.long)
Xte_t = torch.tensor(X_test_sc,  dtype=torch.float32)
yte_t = torch.tensor(y_test,     dtype=torch.long)

batch_size = 1024
train_loader = DataLoader(TensorDataset(Xtr_t, ytr_t), batch_size=batch_size, sampler=sampler, pin_memory=True)
val_loader   = DataLoader(TensorDataset(Xva_t, yva_t), batch_size=4096, shuffle=False, pin_memory=True)
test_loader  = DataLoader(TensorDataset(Xte_t, yte_t), batch_size=4096, shuffle=False, pin_memory=True)

# ===== 5) Mô hình =====
class ResidualBlock(nn.Module):
    def __init__(self, d_in, d_hid, p=0.25):
        super().__init__()
        self.lin1 = nn.Linear(d_in, d_hid)
        self.bn1  = nn.BatchNorm1d(d_hid)
        self.lin2 = nn.Linear(d_hid, d_in)
        self.ln2  = nn.LayerNorm(d_in)
        self.drop = nn.Dropout(p)
    def forward(self, x):
        h = self.drop(torch.relu(self.bn1(self.lin1(x))))
        h = self.lin2(h)
        return torch.relu(self.ln2(x + h))

class ResDNN(nn.Module):
    def __init__(self, in_dim, n_classes):
        super().__init__()
        W = 512
        self.stem = nn.Sequential(nn.Linear(in_dim, W), nn.BatchNorm1d(W), nn.ReLU(), nn.Dropout(0.30))
        self.block1 = ResidualBlock(W, W//2, p=0.30)
        self.block2 = ResidualBlock(W, W//2, p=0.25)
        self.block3 = ResidualBlock(W, W//2, p=0.20)
        self.head   = nn.Linear(W, n_classes)
    def forward(self, x):
        h = self.stem(x)
        h = self.block1(h); h = self.block2(h); h = self.block3(h)
        return self.head(h)

model = ResDNN(num_features, num_classes).to(device)

# ===== 6) CB-Focal Loss + class weights (Effective Number) =====
def cb_class_weights(counts, beta=0.999):
    eff = (1.0 - beta) / (1.0 - np.power(beta, counts + 1e-12))
    w   = eff / eff.mean()
    return torch.tensor(w, dtype=torch.float32, device=device)

class FocalCE(nn.Module):
    def __init__(self, gamma=1.7, weight=None):
        super().__init__()
        self.gamma  = gamma
        self.weight = weight
    def forward(self, logits, target):
        logp = torch.log_softmax(logits, dim=1)
        p    = torch.exp(logp)
        idx  = torch.arange(logits.size(0), device=logits.device)
        pt   = p[idx, target]
        loss = -((1-pt) ** self.gamma) * logp[idx, target]
        if self.weight is not None:
            loss = loss * self.weight[target]
        return loss.mean()

cls_w  = cb_class_weights(counts, beta=0.999)
criterion = FocalCE(gamma=1.7, weight=cls_w)

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=2e-4)

# ===== 7) Lịch LR: warmup + cosine =====
total_epochs  = 120
warmup_epochs = 8
steps_per_epoch = max(1, len(train_loader))
def lr_lambda(current_step):
    warmup_steps = warmup_epochs * steps_per_epoch
    total_steps  = total_epochs * steps_per_epoch
    if current_step < warmup_steps:
        return (current_step + 1) / warmup_steps
    progress = (current_step - warmup_steps) / max(1, (total_steps - warmup_steps))
    return 0.5 * (1 + math.cos(math.pi * progress))
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

# ===== 8) Mixup (giảm khi batch có 7/10) =====
def mixup_dynamic(x, y, base_alpha=0.10):
    alpha = base_alpha
    if any(lbl in (7,10) for lbl in y.tolist()):
        alpha = 0.02
    if alpha <= 0: return x, y, 1.0, None
    lam  = np.random.beta(alpha, alpha)
    perm = torch.randperm(x.size(0), device=x.device)
    x_m  = lam * x + (1 - lam) * x[perm]
    y_perm = y[perm]
    return x_m, y, lam, y_perm

# ===== 9) EMA (float-only) — DEVICE SAFE =====
ema_decay = 0.995
ema_state = {}
for k, v in model.state_dict().items():
    if v.is_floating_point():
        ema_state[k] = v.detach().clone().to(device=v.device, dtype=v.dtype)

def ema_update(model, ema_state, decay: float):
    with torch.no_grad():
        for k, v in model.state_dict().items():
            if not v.is_floating_point():
                continue
            if (ema_state[k].device != v.device) or (ema_state[k].dtype != v.dtype):
                ema_state[k] = ema_state[k].to(device=v.device, dtype=v.dtype)
            ema_state[k].mul_(decay).add_(v.detach(), alpha=1.0 - decay)

def load_ema_state(model, ema_state):
    current = model.state_dict()
    merged = {}
    for k, v in current.items():
        if k in ema_state:
            merged[k] = ema_state[k].to(device=v.device, dtype=v.dtype)
        else:
            merged[k] = v
    model.load_state_dict(merged, strict=True)

# ===== 10) Train + Early-stopping Macro-F1 (VAL) =====
best_f1, best_ema_snapshot, wait, patience = -1.0, None, 0, 20
global_step = 0

for epoch in range(total_epochs):
    model.train()
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        xb_m, ya, lam, yb_ = mixup_dynamic(xb, yb, base_alpha=0.10)
        logits = model(xb_m)
        loss = lam * criterion(logits, ya) + (1 - lam) * criterion(logits, yb_) if yb_ is not None else criterion(logits, ya)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 3.0)
        optimizer.step()
        scheduler.step()
        ema_update(model, ema_state, ema_decay)
        global_step += 1

    # --- Eval VAL với EMA ---
    saved_live = {k: v.detach().clone() for k, v in model.state_dict().items()}
    load_ema_state(model, ema_state)
    model.eval()
    with torch.no_grad():
        logits_val = []
        for xb, _ in val_loader:
            logits_val.append(model(xb.to(device)).cpu())
        logits_val = torch.cat(logits_val, 0).numpy()
    pred_val = logits_val.argmax(1)
    f1_val_all = f1_score(y_val, pred_val, average='macro')
    f1_val_target = f1_score(y_val, pred_val, labels=[7,10], average='macro', zero_division=0)
    f1_val = f1_val_target if OPT_BIAS_FOR_TARGETS else f1_val_all

    if f1_val > best_f1:
        best_f1, wait = f1_val, 0
        best_ema_snapshot = {k: v.detach().cpu().clone() for k, v in ema_state.items()}
    else:
        wait += 1

    model.load_state_dict(saved_live, strict=True)

    if wait >= patience:
        print(f"[Early-stop] epoch {epoch+1} | best VAL Macro-F1 ({'targets' if OPT_BIAS_FOR_TARGETS else 'all'}) = {best_f1:.4f}")
        break

# ===== 11) Load EMA tốt nhất và (tuỳ chọn) finetune head =====
if best_ema_snapshot is not None:
    ema_state = {k: t.clone().to(device) for k, t in best_ema_snapshot.items()}
    load_ema_state(model, ema_state)

if ENABLE_HEAD_FINETUNE:
    for p in model.stem.parameters():   p.requires_grad = False
    for p in model.block1.parameters(): p.requires_grad = False
    for p in model.block2.parameters(): p.requires_grad = False
    for p in model.block3.parameters(): p.requires_grad = False
    optimizer = torch.optim.AdamW(model.head.parameters(), lr=5e-4, weight_decay=1e-4)
    ft_epochs, ft_patience, ft_wait, best_ft = 12, 6, 0, -1.0
    for epoch in range(ft_epochs):
        model.train()
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            logits = model(xb)  # tắt mixup khi finetune head
            loss = criterion(logits, yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.head.parameters(), 3.0)
            optimizer.step()
            ema_update(model, ema_state, ema_decay)
        # eval VAL
        saved_live = {k: v.detach().clone() for k, v in model.state_dict().items()}
        load_ema_state(model, ema_state)
        model.eval()
        with torch.no_grad():
            logits_val = []
            for xb, _ in val_loader:
                logits_val.append(model(xb.to(device)).cpu())
            logits_val = torch.cat(logits_val, 0).numpy()
        pred_val = logits_val.argmax(1)
        f1_val_all = f1_score(y_val, pred_val, average='macro')
        f1_val_target = f1_score(y_val, pred_val, labels=[7,10], average='macro', zero_division=0)
        f1_val = f1_val_target if OPT_BIAS_FOR_TARGETS else f1_val_all
        if f1_val > best_ft:
            best_ft, ft_wait = f1_val, 0
            best_ema_snapshot = {k: v.detach().cpu().clone() for k, v in ema_state.items()}
        else:
            ft_wait += 1
        model.load_state_dict(saved_live, strict=True)
        if ft_wait >= ft_patience:
            print(f"[Finetune head Early-stop] epoch {epoch+1} | best VAL Macro-F1 = {best_ft:.4f}")
            break
    if best_ema_snapshot is not None:
        ema_state = {k: t.clone().to(device) for k, t in best_ema_snapshot.items()}
        load_ema_state(model, ema_state)

model.eval()

# ===== 12) Temperature scaling (VAL) =====
def softmax_np(z):
    z = z - z.max(axis=1, keepdims=True)
    e = np.exp(z); return e / e.sum(axis=1, keepdims=True)

with torch.no_grad():
    logits_val = []
    for xb, _ in val_loader:
        logits_val.append(model(xb.to(device)).cpu())
    logits_val = torch.cat(logits_val, 0).numpy()
    logits_test = []
    for xb, _ in test_loader:
        logits_test.append(model(xb.to(device)).cpu())
    logits_test = torch.cat(logits_test, 0).numpy()

best_T, best_val_f1 = 1.0, -1.0
for T in np.arange(0.5, 3.05, 0.1):
    p_val = softmax_np(logits_val / T)
    pred  = p_val.argmax(1)
    f1_all    = f1_score(y_val, pred, average='macro')
    f1_target = f1_score(y_val, pred, labels=[7,10], average='macro', zero_division=0)
    f1_use    = f1_target if OPT_BIAS_FOR_TARGETS else f1_all
    if f1_use > best_val_f1:
        best_val_f1, best_T = f1_use, float(T)

# ===== 13) (Tuỳ chọn) OvR blend cho 7/10 =====
if ENABLE_OVR_BLEND:
    from sklearn.linear_model import LogisticRegression
    X_ovr = X_tr_in_sc; y_ovr = y_tr_in
    ovr_targets = [7,10]
    lr_models = {}
    for t in ovr_targets:
        y_bin = (y_ovr == t).astype(int)
        lr = LogisticRegression(max_iter=200)
        lr.fit(X_ovr, y_bin)
        lr_models[t] = lr
    ovr_scores_test = {t: lr_models[t].predict_proba(X_test_sc)[:,1] for t in ovr_targets}

# ===== 14) Logit-bias search trên VAL để đẩy 7/10 =====
targets = [7,10]
grid = np.arange(0.0, 1.30, 0.10)
best_b, best_f1_bias = np.zeros(num_classes), -1.0

for b7 in grid:
    for b10 in grid:
        b = np.zeros(num_classes, dtype=np.float32)
        b[7], b[10] = b7, b10
        p_val = softmax_np((logits_val / best_T) + b[None, :])
        pred = p_val.argmax(1)
        f1_all    = f1_score(y_val, pred, average='macro')
        f1_target = f1_score(y_val, pred, labels=targets, average='macro', zero_division=0)
        f1_use    = f1_target if OPT_BIAS_FOR_TARGETS else f1_all
        if f1_use > best_f1_bias:
            best_f1_bias, best_b = f1_use, b.copy()

# ===== 15) Suy luận TEST với T* và bias* (và OvR nếu bật) =====
p_test = softmax_np((logits_test / best_T) + best_b[None, :])

if ENABLE_OVR_BLEND:
    for t in [7,10]:
        s = ovr_scores_test[t]  # [N]
        p_test[:, t] *= (0.5 + 0.5 * s)
    p_test = p_test / p_test.sum(axis=1, keepdims=True)

y_pred = p_test.argmax(1)

# ===== 16) Đánh giá =====
acc = accuracy_score(y_test, y_pred)
macro_f1 = f1_score(y_test, y_pred, average='macro')
cm = confusion_matrix(y_test, y_pred, labels=np.arange(num_classes))
print(f"\nDNN (EMA, T={best_T:.2f}) — Accuracy: {acc:.4f} | Macro-F1: {macro_f1:.4f} | Best VAL-F1(T): {best_val_f1:.4f}")
print(f"Logit-bias best on VAL ({'targets' if OPT_BIAS_FOR_TARGETS else 'all'}): {best_f1_bias:.4f}")
print("\nClassification report:")
print(classification_report(y_test, y_pred, digits=4))

labels_arr = np.arange(num_classes)
prec, rec, f1c, support = precision_recall_fscore_support(
    y_test, y_pred, labels=labels_arr, zero_division=0
)

# FPR per class
fpr_per_class = []
for i in labels_arr:
    TP = cm[i, i]
    FN = cm[i, :].sum() - TP
    FP = cm[:, i].sum() - TP
    TN = cm.sum() - (TP + FN + FP)
    fpr_i = FP / (FP + TN) if (FP + TN) > 0 else 0.0
    fpr_per_class.append(fpr_i)

macro_precision = float(np.mean(prec))
macro_recall    = float(np.mean(rec))
macro_fpr       = float(np.mean(fpr_per_class))

print("\n==== Macro metrics ====")
print(f"Macro-Precision: {macro_precision:.4f}")
print(f"Macro-Recall/TPR: {macro_recall:.4f}")
print(f"Macro-FPR: {macro_fpr:.4f}")

per_class_df = pd.DataFrame({
    "class": labels_arr,
    "precision": prec,
    "recall_TPR": rec,
    "f1": f1c,
    "FPR": fpr_per_class,
    "support": support,
}).round(4)

print("\n==== Per-class metrics (all classes) ====")
print(per_class_df)

[Early-stop] epoch 92 | best VAL Macro-F1 (all) = 0.7078
[Finetune head Early-stop] epoch 7 | best VAL Macro-F1 = 0.7074

DNN (EMA, T=0.50) — Accuracy: 0.6948 | Macro-F1: 0.6801 | Best VAL-F1(T): 0.7074
Logit-bias best on VAL (all): 0.7155

Classification report:
              precision    recall  f1-score   support

           0     0.9941    1.0000    0.9971       169
           1     0.8343    0.8343    0.8343       169
           2     0.7792    0.7101    0.7430       169
           3     0.5059    0.2544    0.3386       169
           4     0.5065    0.2308    0.3171       169
           5     0.6623    0.9053    0.7650       169
           6     0.4664    0.7811    0.5841       169
           7     0.6032    0.8994    0.7221       169
           8     0.8908    0.6272    0.7361       169
           9     0.5586    0.4793    0.5159       169
          10     0.6000    0.6213    0.6105       169
          11     1.0000    0.9941    0.9970       169

    accuracy                    

In [9]:
# Save ResDNN checkpoint (compatible with ResDNNWrapper.from_checkpoint)
import os, torch

os.makedirs('./models', exist_ok=True)

checkpoint = {
    'state_dict': model.state_dict(),
    'scaler': scaler,
    'in_dim': num_features,
    'n_classes': num_classes,
}

save_path = './models/framework_resdnn_TVAE.pth'
torch.save(checkpoint, save_path)
print(f"ResDNN checkpoint saved to {save_path}")
print(f"  in_dim={num_features}, n_classes={num_classes}")

ResDNN checkpoint saved to ./models/framework_resdnn_TVAE.pth
  in_dim=66, n_classes=12
